# Wave Data Downloader — Marconi Beach, Cape Cod

**Author:** Chelsea Volpano  
**Email:** cvolpano@contractor.usgs.gov | cvolpano@gmail.com  
**Date:** 2026-06-24  
**Created using Claude Sonnet 4.6**

---

Downloads wave data (Oct 2024 – Mar 2025) from:
- **WIS Hindcast** Station 63064 via the WIS Portal API (Oct–Dec 2024)
- **NDBC Buoy 44020** (Nantucket Sound, nearshore — full period)
- **NDBC Buoy 44018** (SE Cape Cod, open Atlantic — 2024 only)

In [46]:
import requests
import pandas as pd
import numpy as np
from io import StringIO
from pathlib import Path

In [47]:
# ── Configuration ─────────────────────────────────────────────────────────────
OUT_DIR = Path("wave_data")
OUT_DIR.mkdir(exist_ok=True)

START = "2024-10-01"
END   = "2025-03-31"

## 1. WIS Hindcast — Station 63064

Uses the WIS Portal API at `wisportal.erdc.dren.mil`, passing the THREDDS OPeNDAP
dataset URL as the `url` parameter. Returns CSV. Data available through end of 2024.

In [51]:
WIS_STATION    = "63064"
WIS_PORTAL_API = "https://wisportal.erdc.dren.mil/data/api/station/data"
WIS_THREDDS    = "https://chldata.erdc.dren.mil/thredds/dodsC/wis/Atlantic"
WIS_VARIABLES  = "waveHs,waveTp,waveTm,waveMeanDirection"

def fetch_wis(station: str, start: str, end: str) -> pd.DataFrame | None:
    dataset_url = f"{WIS_THREDDS}/ST{station}/ST{station}.nc4"
    params = {
        "url":       dataset_url,
        "time":      f"{start}T00:00:00/{end}T23:59:59",
        "format":    "csv",
        "variables": WIS_VARIABLES,
    }
    # Show the full composed URL for verification
    req = requests.Request("GET", WIS_PORTAL_API, params=params).prepare()
    print(f"  Full request URL:\n  {req.url}\n")
    try:
        r = requests.get(WIS_PORTAL_API, params=params, timeout=120)
        r.raise_for_status()
    except Exception as e:
        print(f"  ⚠ Request failed: {e}")
        return None

    lines = [l for l in r.text.splitlines() if not l.strip().startswith("#")]
    df = pd.read_csv(StringIO("\n".join(lines)))
    print(f"  Columns: {list(df.columns)}")

    for dt_col in ["time", "datetime", "date", "TIME", "DATE"]:
        if dt_col in df.columns:
            df["datetime"] = pd.to_datetime(df[dt_col])
            if dt_col != "datetime":
                df = df.drop(columns=[dt_col])
            break
    else:
        print(f"  ⚠ Could not find datetime column. Columns: {list(df.columns)}")
        return df

    df = df.set_index("datetime")
    print(f"  ✓ Retrieved {len(df)} rows")
    return df

print("── WIS Hindcast ──")
wis_clip = fetch_wis(WIS_STATION, START, "2024-12-31")

if wis_clip is not None:
    wis_out = OUT_DIR / f"WIS_ST{WIS_STATION}_{START[:7]}_to_2024-12.csv"
    wis_clip.to_csv(wis_out)
    print(f"  ✓ Saved {len(wis_clip)} rows → {wis_out}")
else:
    print("  ⚠ No WIS data retrieved.")

── WIS Hindcast ──
  Full request URL:
  https://wisportal.erdc.dren.mil/data/api/station/data?url=https%3A%2F%2Fchldata.erdc.dren.mil%2Fthredds%2FdodsC%2Fwis%2FAtlantic%2FST63064%2FST63064.nc4&time=2024-10-01T00%3A00%3A00%2F2024-12-31T23%3A59%3A59&format=csv&variables=waveHs%2CwaveTp%2CwaveTm%2CwaveMeanDirection

  Columns: ['time', 'lat', 'lon', 'waveMeanDirection', 'waveTp', 'waveTm', 'waveHs']
  ✓ Retrieved 2205 rows
  ✓ Saved 2205 rows → wave_data/WIS_ST63064_2024-10_to_2024-12.csv


In [52]:
wis_clip.head()

,lat,lon,waveMeanDirection,waveTp,waveTm,waveHs
datetime,,,,,,
2024-10-01 01:00:00,41.916672,-69.75,80.226562,9.437500,8.375000,0.898438
2024-10-01 02:00:00,41.916672,-69.75,80.500000,9.468750,8.398438,0.890625
2024-10-01 03:00:00,41.916672,-69.75,80.773438,9.476562,8.421875,0.882812
2024-10-01 04:00:00,41.916672,-69.75,81.039062,9.460938,8.445312,0.882812
2024-10-01 05:00:00,41.916672,-69.75,81.273438,9.445312,8.468750,0.875000


## 2. NDBC Buoy Data

Fetches standard meteorological (stdmet) data from NOAA NDBC historical archive.  
Key wave columns: **WVHT** (Hs, m), **DPD** (dominant period, s), **APD** (avg period, s), **MWD** (mean direction, °).

| Buoy | Location | Notes |
|------|----------|-------|
| 44020 | Nantucket Sound | Primary — full period available |
| 44018 | SE Cape Cod (~60 nm offshore) | 2024 only |

In [48]:
NDBC_BASE  = "https://www.ndbc.noaa.gov/data/historical/stdmet"
NDBC_BUOYS = {
    "44020": "Nantucket Sound (primary)",
    "44018": "SE Cape Cod (open Atlantic, 2024 only)",
}
NDBC_YEARS = [2024, 2025]

def fetch_ndbc(buoy: str, year: int) -> pd.DataFrame | None:
    import gzip, shutil
    url = f"{NDBC_BASE}/{buoy}h{year}.txt.gz"
    gz_path  = OUT_DIR / f"NDBC_{buoy}_{year}.txt.gz"
    txt_path = OUT_DIR / f"NDBC_{buoy}_{year}.txt"
    print(f"  Fetching NDBC {buoy} {year}: {url}")
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        gz_path.write_bytes(r.content)
        with gzip.open(gz_path, "rb") as f_in, open(txt_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        gz_path.unlink()  # clean up .gz
        df = pd.read_csv(txt_path, sep=r"\s+", low_memory=False,
                         na_values=["99", "999", "9999", "99.0", "999.0"])
        txt_path.unlink()  # clean up .txt
    except Exception as e:
        print(f"  ⚠ Could not read {buoy} {year}: {e}")
        return None

    df.columns = df.columns.str.strip().str.lstrip("#")
    # Drop units row if present
    if df.iloc[0].astype(str).str.contains("yr|#yr", case=False).any():
        df = df.iloc[1:].reset_index(drop=True)
    print(f"  Columns: {list(df.columns)}")
    # Normalise year column name and value
    if "YY" in df.columns and "YYYY" not in df.columns:
        df = df.rename(columns={"YY": "YYYY"})
    df["YYYY"] = df["YYYY"].astype(int)
    # Fix 2-digit years (e.g. 24 → 2024) and 4-digit years stored as 1924 → 2024
    df["YYYY"] = df["YYYY"].apply(lambda y: y + 2000 if y < 100 else (y + 100 if y < 2000 else y))
    print(f"  YYYY sample after fix: {df['YYYY'].iloc[0]}")

    try:
        mc = "mm" if "mm" in df.columns else ("MM" if "MM" in df.columns else None)
        if mc:
            df["datetime"] = pd.to_datetime(
                df["YYYY"].astype(str) + df["MM"].astype(str).str.zfill(2) +
                df["DD"].astype(str).str.zfill(2) + df["hh"].astype(str).str.zfill(2) +
                df[mc].astype(str).str.zfill(2), format="%Y%m%d%H%M", errors="coerce")
        else:
            df["datetime"] = pd.to_datetime(
                df["YYYY"].astype(str) + df["MM"].astype(str).str.zfill(2) +
                df["DD"].astype(str).str.zfill(2) + df["hh"].astype(str).str.zfill(2),
                format="%Y%m%d%H", errors="coerce")
    except KeyError:
        print(f"  ⚠ Unexpected columns for {buoy} {year}: {list(df.columns)}")
        return None

    print(f"  datetime sample: {df['datetime'].head().tolist()}")
    print(f"  datetime nulls: {df['datetime'].isna().sum()} of {len(df)}")
    df = df.set_index("datetime")
    print(f"  index range: {df.index.min()} → {df.index.max()}")
    wave_cols = [c for c in ["WVHT", "DPD", "APD", "MWD"] if c in df.columns]
    print(f"  wave cols found: {wave_cols}")
    return df[wave_cols]

print("── NDBC Buoys ──")
ndbc_dfs = {}
for buoy, desc in NDBC_BUOYS.items():
    print(f"\n  Buoy {buoy} — {desc}")
    frames = [fetch_ndbc(buoy, y) for y in NDBC_YEARS]
    frames = [f for f in frames if f is not None]
    if not frames:
        print(f"  ⚠ No data retrieved for buoy {buoy}.")
        continue
    combined = pd.concat(frames)
    combined = combined[~combined.index.duplicated(keep="first")]
    clipped  = combined.loc[START:END].copy()
    ndbc_dfs[buoy] = clipped
    out_path = OUT_DIR / f"NDBC_{buoy}_{START[:7]}_to_{END[:7]}.csv"
    clipped.to_csv(out_path)
    print(f"  ✓ Saved {len(clipped)} rows → {out_path}")

── NDBC Buoys ──

  Buoy 44020 — Nantucket Sound (primary)
  Fetching NDBC 44020 2024: https://www.ndbc.noaa.gov/data/historical/stdmet/44020h2024.txt.gz
  Columns: ['YY', 'MM', 'DD', 'hh', 'mm', 'WDIR', 'WSPD', 'GST', 'WVHT', 'DPD', 'APD', 'MWD', 'PRES', 'ATMP', 'WTMP', 'DEWP', 'VIS', 'TIDE']
  YYYY sample after fix: 2024
  datetime sample: [Timestamp('2024-01-01 00:00:00'), Timestamp('2024-01-01 00:10:00'), Timestamp('2024-01-01 00:20:00'), Timestamp('2024-01-01 00:30:00'), Timestamp('2024-01-01 00:40:00')]
  datetime nulls: 0 of 52703
  index range: 2024-01-01 00:00:00 → 2024-12-31 23:50:00
  wave cols found: ['WVHT', 'DPD', 'APD', 'MWD']
  Fetching NDBC 44020 2025: https://www.ndbc.noaa.gov/data/historical/stdmet/44020h2025.txt.gz
  Columns: ['YY', 'MM', 'DD', 'hh', 'mm', 'WDIR', 'WSPD', 'GST', 'WVHT', 'DPD', 'APD', 'MWD', 'PRES', 'ATMP', 'WTMP', 'DEWP', 'VIS', 'TIDE']
  YYYY sample after fix: 2025
  datetime sample: [Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-01 00:10:00

In [ ]:
ndbc_dfs["44020"].head()

## 3. Summary Statistics

In [49]:
if wis_clip is not None:
    print("── WIS ST63064 ──")
    print(wis_clip.describe().round(2))

print("\n── NDBC 44020 ──")
print(ndbc_dfs["44020"].describe().round(2))


── NDBC 44020 ──
         WVHT    DPD    APD   MWD
count   26208  26208  26208  7034
unique    161     41    243   359
top     99.00  99.00  99.00   263
freq    17522  19148  17522    96
